## Prepare questions from NACC for testing GRPO model

Created by: Sahana Kowshik

Date: 05/09/2025

In [1]:
import pandas as pd
import json
import random
import numpy as np

In [2]:
directory_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc"

In [3]:
test = pd.read_csv(f"{directory_path}/training_data/testing_data_grpo/test_summary.csv")
train = pd.read_csv(f"{directory_path}/training_data/training_data_grpo/train_summary.csv")

/scratch/9210280.1.cbm.q/ipykernel_2361871/3264798635.py:1: DtypeWarning: Columns (20,22,24,28,43,45,50,93,94,95,96,97,98,99,100,101,102,103,104,105,164,175,178,216,219,221,223,225,227,229,231,233,235,237,239,241,243,245,398,400,420,422,431,444,453,571,599,607,663,679,696,699,716,727,733,808,825,949,957,958,959,960,998,1017,1022,1192,1196,1199,1395,1397,1399,1400,1402,1409,1411,1413,1414,1421,1423,1425,1427,1428,1435,1450,1464,1478,1492,1494,1530,1546,1548,1550,1552,1554,1556,1558,1560,1562,1564,1566,1568,1570,1572,1574,1576,1578,1580,1582,1584,1586,1588,1590,1592,1594,1596,1598,1600,1650,1651,1653,1654,1657,1658,1661,1662,1665,1666,1669,1670,1744,1803,1812,1814,1816,1818,1829,1831,1833,1841,1843,1845,1847,1855,1857,1859,1861,1887) have mixed types. Specify dtype option on import or set low_memory=False.
  test = pd.read_csv(f"{directory_path}/training_data/testing_data_grpo/test_summary.csv")


In [4]:
set(train['ID']).intersection(set(test['NACCID']))

set()

In [6]:
# Add ETPR variables
def add_etpr_labels(row):
    value = row['NACCETPR']
    
    # NC
    if value == 88:
        row['ETPR'] = 'NC'
    
    # AD
    elif value == 1:
        row['ETPR'] = 'AD'

    # LBD
    elif value == 2:
        row['ETPR'] = 'LBD'

    # VD
    elif value == 8:
        row['ETPR'] = 'VD'

    # FTLD and its variants, including CBD and PSP, with or without ALS
    elif value in [4, 5, 6, 7]:
        row['ETPR'] = 'FTLD'
    
    # SEF: seizure, epilepsy, etc.
    elif value in [17, 18, 23, 26, 27, 28, 29]:
        row['ETPR'] = 'SEF'

    # Psychiatric conditions
    elif value in [19, 20, 21, 22, 24, 25]:
        row['ETPR'] = 'PSY'

    # Other degenerative
    elif value not in [30, 88, 99]:
        row['ETPR'] = 'ODE'

    else:
        row['ETPR'] = np.NaN

    return row


test = test.apply(add_etpr_labels, axis=1)

In [7]:
len(test[~test['ETPR'].isna()])

5626

In [8]:
sum(dict(test['ETPR'].value_counts()).values())

5626

In [9]:
test['ETPR'].value_counts()

ETPR
AD      3219
NC       951
FTLD     759
LBD      410
VD       148
ODE       93
SEF       34
PSY       12
Name: count, dtype: int64

In [10]:
# Function to add NP ground truth columns
def add_mci_ground_truth(row):
    # Alzheimer's Disease
    if row['NACCTMCI'] in [1, 2]:
        row['MCI'] = 0
    elif row['NACCTMCI'] in [3, 4]:
        row['MCI'] = 1
    elif row['NACCTMCI'] in [8]:
        row['MCI'] = 2

    return row

# Apply the function across the training set
test = test.apply(add_mci_ground_truth, axis=1)

In [11]:
test['MCI'].value_counts()

MCI
2    5320
0     380
1     112
Name: count, dtype: int64

In [12]:
test['ID'] = test['NACCID']
test.drop(['NACCID'], axis=1, inplace=True)

/scratch/9210280.1.cbm.q/ipykernel_2361871/3636380100.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test['ID'] = test['NACCID']


In [14]:
columns = ['ID', 'patient_summary', 'diag_summary', 'visit_summary',  'question', 'options', 'ground_truth', 'ground_truth_text', 'COHORT']

## Neuropathology

In [15]:
# test_np = test[((test['NACCLEWY'].isin([0,1,2,3,4])) & (test['NPFTDTAU'].isin([0,1])) & (test['NPADNC'].isin([0,1,2,3])))].sample(frac=1, random_state=0).reset_index(drop=True)

In [16]:
test_np = test[
    (test['NACCUDSD'] != 1) &
    (test['NACCLEWY'].isin([0, 1, 2, 3, 4])) &
    (
        (test['NPFTDTAU'].isin([0]) & test['NPFTDTDP'].isin([0])) | 
        test['NPFTDTAU'].isin([1]) | 
        test['NPFTDTDP'].isin([1])
    ) &
    (test['NPADNC'].isin([0, 1, 2, 3])) &
    (test['VISITGAP'] <= 3)
].sample(frac=1, random_state=42).reset_index(drop=True)

print(len(test_np))


668


In [17]:
# Function to add NP ground truth columns
def add_ground_truth(row):
    # Alzheimer's Disease
    if row['NPADNC'] in [2, 3]:
        row['NP_AD'] = 1
        row['NP_AVAIL'] = 1
    elif row['NPADNC'] in [0, 1]:
        row['NP_AD'] = 0
        row['NP_AVAIL'] = 1

    # Lewy Body Disease
    if row['NACCLEWY'] in [1, 2, 3, 4]:
        row['NP_LBD'] = 1
        row['NP_AVAIL'] = 1
    elif row['NACCLEWY'] == 0:
        row['NP_LBD'] = 0
        row['NP_AVAIL'] = 1

    # Frontotemporal Degeneration (FTLD-tau or other)
    if (row['NPFTDTAU'] == 1) | (row['NPFTDTDP'] == 1):
        row['NP_FTD'] = 1
        row['NP_AVAIL'] = 1
    elif (row['NPFTDTAU'] == 0) & (row['NPFTDTDP'] == 0):
        row['NP_FTD'] = 0
        row['NP_AVAIL'] = 1

    return row

In [18]:
# Apply the function across the training set
test_np = test_np.apply(add_ground_truth, axis=1)

# Check distribution of available neuropathology labels
test_np['NP_AVAIL'].value_counts()

NP_AVAIL
1    668
Name: count, dtype: int64

In [19]:
import random, string
import pandas as pd

# Neuropathology Question Variants
np_question_variants = [
    "Which of the following pathologies are causing the patient's cognitive impairment based on the clinical information provided?",
    "Using the provided clinical information, determine the pathological causes of the patient's cognitive decline from the options listed below.",
    "Based on the clinical data, identify the neuropathologies responsible for the patient's cognitive impairment by selecting from the given choices.",
    "From the options below, select the disease pathologies causing cognitive impairment as indicated by the clinical information.",
    "Review the provided clinical information and choose the pathologies underlying the patient's cognitive symptoms from the available options."
]

# Pull the bare answer texts into a list – easier to shuffle
NP_OPTION_TEXTS = [
    "Alzheimer's disease pathology (AD) only",
    "Lewy body pathology (LBD) only",
    "Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD) only",
    "Alzheimer's disease pathology (AD) and Lewy body pathology (LBD)",
    "Alzheimer's disease pathology (AD) and Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD)",
    "Lewy body pathology (LBD) and Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD)",
    "Alzheimer's disease pathology (AD), Lewy body pathology (LBD) and Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD)",
    "No listed option is correct",
]

# map each neuropathology pattern → the **text** of the correct option
NP_ANSWER_MAP = {
    (1, 1, 1): "Alzheimer's disease pathology (AD), Lewy body pathology (LBD) and Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD)",
    (0, 1, 1): "Lewy body pathology (LBD) and Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD)",
    (1, 0, 1): "Alzheimer's disease pathology (AD) and Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD)",
    (1, 1, 0): "Alzheimer's disease pathology (AD) and Lewy body pathology (LBD)",
    (0, 0, 1): "Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD) only",
    (0, 1, 0): "Lewy body pathology (LBD) only",
    (1, 0, 0): "Alzheimer's disease pathology (AD) only",
    (0, 0, 0): "No listed option is correct",
}


def add_np_question(row, *, seed=None):
    """
    Adds three new columns to each row:
    ─ question : one of the phrasing variants           (string)
    ─ options  : the shuffled, letter-prefixed choices  (multiline string)
    ─ ground_truth : the **letter** (A-H) of the correct choice (string)
    """
    rng = random.Random(seed if seed is not None else row.name)  # reproducible per-row if you like

    # 1️⃣ Choose a question at random
    row["question"] = rng.choice(np_question_variants)

    # 2️⃣ Shuffle the options and assign fresh letters
    shuffled = NP_OPTION_TEXTS.copy()
    rng.shuffle(shuffled)
    letters = list(string.ascii_uppercase[: len(shuffled)])
    row["options"] = "\n".join(f"{ltr}. {opt}" for ltr, opt in zip(letters, shuffled))

    # 3️⃣ Work out which *text* is correct for this patient
    combo = (row["NP_AD"], row["NP_LBD"], row["NP_FTD"])
    correct_text = NP_ANSWER_MAP[combo]

    # 4️⃣ Convert that to the new letter after shuffling
    correct_letter = letters[shuffled.index(correct_text)]
    row["ground_truth"] = correct_letter
    
    row['ground_truth_text'] = correct_text

    return row


In [20]:
# Apply question and label assignment
test_np = test_np.apply(add_np_question, axis=1)

# Check ground truth distribution
test_np['ground_truth'].value_counts()

ground_truth
E    101
G     96
A     87
F     84
H     82
B     81
D     74
C     63
Name: count, dtype: int64

In [21]:
# Update set of NACCIDs used to avoid duplication later
np_ids = set(test_np['ID'])

# Optional: Check distribution of diagnosis codes
test_np['NACCUDSD'].value_counts()

NACCUDSD
4    581
3     87
Name: count, dtype: int64

In [22]:
# Select final columns to keep
test_np = test_np[columns]  # Make sure 'columns' is defined somewhere earlier
test_np['Q_TYPE'] = 'Neuropath'

In [26]:
test_np.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/with_summary/test_np.csv", index=False)

## Amylpet

In [23]:
# # Filter training datA. exclude patients already used in NP task and keep only those with valid amyloid PET results
# test_pet = (
#     test
#     .loc[
#         (test['AMYLPET'].isin([0, 1]))
#     ]
#     .sample(frac=1, random_state=42)  # Shuffle all rows
#     .reset_index(drop=True)
# )

# # Check the distribution of amyloid PET labels
# test_pet['AMYLPET'].value_counts()

In [24]:
# # Question variants for amyloid PET classification
# pet_question_variants = [
#     "Identify if the patient is amyloid positive based on the provided information.",
#     "Determine whether this patient is amyloid positive using the given data.",
#     "Analyze the patient information to identify amyloid positivity.",
#     "Based on the details provided, identify if the patient shows amyloid positivity.",
#     "Using the provided patient data, identify whether the patient is amyloid positive."
# ]

# # Multiple-choice answer options
# pet_options = """A. Yes
# B. No"""

In [25]:
# # Keys related to amyloid biomarkers that should be removed from the patient summary
# pet_keys_remove = [
#     'Abnormally elevated amyloid on PET',
#     'Abnormally low amyloid in CSF',
#     'FDG-PET pattern of AD',
#     'Tau PET evidence for AD',
#     'Abnormally elevated CSF Tau or pTau'
# ]

In [26]:
# # Function to add question, options, ground truth, and modify patient summary
# def add_pet_question(row):
#     # Set ground truth answer
#     if row['AMYLPET'] == 1:
#         row['ground_truth'] = "A"  # Amyloid positive
#     elif row['AMYLPET'] == 0:
#         row['ground_truth'] = "B"  # Amyloid negative

#     # Randomly assign one of the question variants
#     row['question'] = random.choice(pet_question_variants)

#     # Remove specific amyloid-related keys from the patient summary
#     json_file = json.loads(row['patient_summary'])
#     for key in pet_keys_remove:
#         if key in json_file.get('Imaging evidence', {}):
#             json_file['Imaging evidence'].pop(key)
            
#         if key in json_file.get('CSF evidence', {}):
#             json_file['CSF evidence'].pop(key)
    
#     # Update patient summary
#     row['patient_summary'] = json.dumps(json_file, indent=4)
    
#     # Set options
#     row["options"] = pet_options

#     return row

In [27]:
# # Apply the function across the PET training dataset
# test_pet = test_pet.apply(add_pet_question, axis=1)

# # Quick check: How many times each question variant was picked
# print(test_pet['question'].value_counts())

# # Check distribution of amyloid PET labels
# print(test_pet['AMYLPET'].value_counts())

In [28]:
# print(test_pet['ground_truth'].value_counts())

In [29]:
# # Update set of NACCIDs used to avoid reusing subjects in other tasks
# pet_ids = set(test_pet['ID'])

# # Optional: Check cognitive diagnosis distribution
# test_pet['NACCUDSD'].value_counts()

In [30]:
# # Select only necessary columns
# test_pet = test_pet[columns]
# test_pet['Q_TYPE'] = 'Amyloid PET'

In [31]:
# print(test[test['NACCID'] == 'NACC867075'].iloc[0]['patient_summary'])

In [32]:
# for key in pet_keys_remove:
#     for i, row in test_pet.iterrows():
#         if key in row['patient_summary']:
#             print(i, key)
#             # break

In [33]:
# test_pet.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/with_questions/test_pet.csv", index=False)

## Datscan

In [34]:
# test_dat = test[(~test['NACCID'].isin(np_ids.union(pet_ids))) & (test['DATSCAN'].isin([0,1]))].apply(lambda x: x.sample(n=200, random_state=42)).reset_index(drop=True)

In [35]:
# # Filter training datA. exclude subjects already used for NP and PET tasks,
# # and keep only those with valid DATScan results
# test_dat = (
#     test
#     .loc[
#         (test['DATSCAN'].isin([0, 1]))
#     ]
#     .sample(frac=1, random_state=42)  # Shuffle all rows
#     .reset_index(drop=True)
# )

# # Check the distribution of DATScan labels
# test_dat['DATSCAN'].value_counts()


In [36]:
# # Question variants for DATScan classification
# dat_question_variants = [
#     "Classify if the patient is DatScan positive based on the provided information.",
#     "Using the given data, determine whether the patient is DatScan positive.",
#     "Analyze the provided information to classify the patient's DatScan status.",
#     "Based on the information given, classify whether the patient is DatScan positive.",
#     "From the provided patient details, identify if the patient is DatScan positive."
# ]

# # Multiple-choice answer options
# dat_options = """A. Yes
# B. No"""

In [37]:
# # Keys related to DATScan that should be removed from the patient summary
# dat_keys_remove = [
#     'Dopamine transporter scan (DATscan) evidence for Lewy body disease'
# ]


In [38]:
# # Function to add question, options, ground truth, and modify patient summary for DAT task
# def add_dat_question(row):
#     # Set ground truth answer
#     if row['DATSCAN'] == 1:
#         row['ground_truth'] = "A"  # DATScan positive
#     elif row['DATSCAN'] == 0:
#         row['ground_truth'] = "B"  # DATScan negative

#     # Randomly assign one of the question variants
#     row['question'] = random.choice(dat_question_variants)

#     # Remove specific DATScan-related keys from the patient summary
#     json_file = json.loads(row['patient_summary'])
#     for key in dat_keys_remove:
#         if key in json_file.get('Imaging evidence', {}):
#             json_file['Imaging evidence'].pop(key)
    
#     # Update patient summary
#     row['patient_summary'] = json.dumps(json_file, indent=4)
    
#     # Set options
#     row["options"] = dat_options

#     return row


In [39]:
# # Apply the function across the DAT training dataset
# test_dat = test_dat.apply(add_dat_question, axis=1)

# # Check how many times each question variant was selected
# print(test_dat['question'].value_counts())

# # Check distribution of DATScan labels
# print(test_dat['NACCUDSD'].value_counts())

In [40]:
# print(test_dat['ground_truth'].value_counts())

In [41]:
# # Update set of NACCIDs used to avoid reusing subjects in other tasks
# dat_ids = set(test_dat['ID'])

In [42]:
# # Select only the necessary columns
# test_dat = test_dat[columns]
# test_dat['Q_TYPE'] = 'DATSCAN'

In [43]:
# for key in dat_keys_remove:
#     for i, row in test_dat.iterrows():
#         if key in row['patient_summary']:
#             print(i, key)
#             # break

In [44]:
# test_dat.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/with_questions/test_dat.csv", index=False)

## NACCTMCI

In [27]:
test['MCI'].value_counts()

MCI
2    5320
0     380
1     112
Name: count, dtype: int64

In [28]:
# Sample MCI patients, excluding those already selected for NP, PET, or DAT tasks
test_mci = (
    test
    .groupby('MCI', group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), 380), random_state=42))  # Safe sampling
    .sample(frac=1, random_state=42)  # Shuffle the sampled rows
    .reset_index(drop=True)
)

# Check the number of sampled patients per MCI subtype
test_mci['MCI'].value_counts()

/scratch/9210280.1.cbm.q/ipykernel_2361871/1829277309.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), 380), random_state=42))  # Safe sampling


MCI
2    380
0    380
1    112
Name: count, dtype: int64

In [29]:
test_mci['NACCUDSD'].value_counts()

NACCUDSD
3    492
4    315
1     65
Name: count, dtype: int64

In [30]:
import random, string

# Question variants for MCI subtype classification
mci_question_variants = [
    "Determine, if applicable, the correct subtype of mild cognitive impairment (MCI) for this patient based on the provided information.",
    "Based on the patient's data, identify the appropriate mild cognitive impairment (MCI) subtype, if applicable.",
    "Using the information given, classify the patient's mild cognitive impairment (MCI) subtype, if applicable.",
    "If applicable, identify the subtype of mild cognitive impairment based on the provided patient details.",
    "Analyze the patient's information to identify the correct mild cognitive impairment (MCI) subtype, if applicable."
]


# MCI subtype answer options (original order)
MCI_OPTION_TEXTS = [
    "Amnestic MCI",
    "Non-amnestic MCI",
    "Not applicable (no diagnosis of MCI)",
]

# Mapping NACCTMCI code → correct answer text
MCI_ANSWER_MAP = {
    0: "Amnestic MCI",
    1: "Non-amnestic MCI",
    2: "Not applicable (no diagnosis of MCI)",
}

def add_mci_question(row, *, seed=None):
    rng = random.Random(seed if seed is not None else row.name)

    # 1️⃣ Randomly choose a question phrasing
    row['question'] = rng.choice(mci_question_variants)

    # 2️⃣ Shuffle the answer texts and assign letters
    shuffled = MCI_OPTION_TEXTS.copy()
    rng.shuffle(shuffled)
    letters = list(string.ascii_uppercase[:len(shuffled)])
    row['options'] = "\n".join(f"{ltr}. {opt}" for ltr, opt in zip(letters, shuffled))

    # 3️⃣ Get the correct text for this patient's subtype
    correct_text = MCI_ANSWER_MAP.get(row['MCI'], None)

    # 4️⃣ Assign ground truth letter if valid
    if correct_text in shuffled:
        correct_letter = letters[shuffled.index(correct_text)]
        row['ground_truth'] = correct_letter
    else:
        row['ground_truth'] = None  # in case NACCTMCI is missing or invalid
        
    row['ground_truth_text'] = correct_text

    return row


In [31]:
# Apply the function to the MCI training set
test_mci = test_mci.apply(add_mci_question, axis=1)

In [32]:
# Quick checks
print(test_mci['ground_truth'].value_counts())
print(test_mci['ground_truth_text'].value_counts())
print(test_mci['question'].value_counts())
print(test_mci['NACCUDSD'].value_counts())

ground_truth
A    307
C    304
B    261
Name: count, dtype: int64
ground_truth_text
Not applicable (no diagnosis of MCI)    380
Amnestic MCI                            380
Non-amnestic MCI                        112
Name: count, dtype: int64
question
Determine, if applicable, the correct subtype of mild cognitive impairment (MCI) for this patient based on the provided information.    179
Based on the patient's data, identify the appropriate mild cognitive impairment (MCI) subtype, if applicable.                           176
If applicable, identify the subtype of mild cognitive impairment based on the provided patient details.                                 174
Analyze the patient's information to identify the correct mild cognitive impairment (MCI) subtype, if applicable.                       173
Using the information given, classify the patient's mild cognitive impairment (MCI) subtype, if applicable.                             170
Name: count, dtype: int64
NACCUDSD
3    492
4    

In [33]:
# Update set of used IDs to avoid reusing these patients elsewhere
mci_ids = set(test_mci['ID'])

In [34]:
# Select only the columns needed for modeling
test_mci = test_mci[columns]
test_mci['Q_TYPE'] = 'MCI subtype'

In [36]:
test_mci.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/with_summary/test_mci.csv", index=False)

## NACCETPR

In [37]:
# Filter and sample patients for etiology prediction task,
# excluding those already selected for NP, PET, DAT, or MCI tasks
test_etpr = (
    test
    .loc[
        ~(test['ETPR'].isna())  
    ]
    # .groupby('ETPR', group_keys=False)
    # .apply(lambda x: x.sample(n=min(len(x), 1000), random_state=42))  # Sample up to 1000
    .sample(frac=1, random_state=42)  # Shuffle the sampled rows
    .reset_index(drop=True)
)

# Check sampled distribution
test_etpr['ETPR'].value_counts()

ETPR
AD      3219
NC       951
FTLD     759
LBD      410
VD       148
ODE       93
SEF       34
PSY       12
Name: count, dtype: int64

In [38]:
import random, string

etiology_question_variants = [
    "Identify the primary etiologic diagnosis of the patient using the information provided, if applicable.",
    "Identify the primary etiology of the patient's cognitive impairment using the information provided, if applicable.",
    "Based on the clinical details, determine the main cause of the patient's cognitive impairment, if applicable.",
    "Using the information provided, identify the primary cause of the patient's cognitive decline, if applicable.",
    "Determine the primary etiology underlying the patient's cognitive impairment based on the provided information, if applicable."
]

# Raw etiology answer texts in original order
ETIOLOGY_OPTION_TEXTS = [
    "Alzheimer's disease (AD)",
    "Lewy body disease (LBD)",
    "Frontotemporal lobar degeneration and its variants, including primary progressive aphasia, corticobasal degeneration and progressive supranuclear palsy, and with or without amyotrophic lateral sclerosis (FTLD)",
    "Vascular brain injury or vascular dementia including stroke (VD)",
    "Systemic and environmental factors including infectious diseases (HIV included), metabolic, substance abuse / alcohol, medications, systemic disease and delirium (SEF)",
    "Psychiatric conditions including schizophrenia, depression, bipolar disorder, anxiety and posttraumatic stress disorder (PSY)",
    "Other (Multiple system atrophy, Essential tremor, Down syndrome, Huntington's disease, Prion disease, Traumatic brain injury, Normal-pressure hydrocephalus, Epilepsy, CNS neoplasm, etc)",
    "Not applicable (no cognitive impairment)",
]

# Mapping of ETPR code to correct answer text
ETPR_ANSWER_MAP = {
    'AD': ETIOLOGY_OPTION_TEXTS[0],
    'LBD': ETIOLOGY_OPTION_TEXTS[1],
    'FTLD': ETIOLOGY_OPTION_TEXTS[2],
    'VD': ETIOLOGY_OPTION_TEXTS[3],
    'SEF': ETIOLOGY_OPTION_TEXTS[4],
    'PSY': ETIOLOGY_OPTION_TEXTS[5],
    'ODE': ETIOLOGY_OPTION_TEXTS[6],
    'NC': ETIOLOGY_OPTION_TEXTS[7],  # fallback/default
}

def add_etpr_question(row, *, seed=None):
    rng = random.Random(seed if seed is not None else row.name)

    # 1️⃣ Randomly select a question variant
    row['question'] = rng.choice(etiology_question_variants)

    # 2️⃣ Shuffle options and assign fresh letters
    shuffled = ETIOLOGY_OPTION_TEXTS.copy()
    rng.shuffle(shuffled)
    letters = list(string.ascii_uppercase[:len(shuffled)])
    row['options'] = "\n".join(f"{ltr}. {opt}" for ltr, opt in zip(letters, shuffled))

    # 3️⃣ Get correct text from ETPR code (default to "Not applicable" if unknown)
    correct_text = ETPR_ANSWER_MAP.get(row['ETPR'])

    if correct_text is None:
        raise ValueError(f"Invalid ETPR value: {row['ETPR']}")


    # 4️⃣ Get ground truth letter from shuffled list
    if correct_text in shuffled:
        row['ground_truth'] = letters[shuffled.index(correct_text)]
    else:
        row['ground_truth'] = None  # shouldn't happen unless text mismatch
        
    row['ground_truth_text'] = correct_text

    return row


In [39]:
# Apply the function to the MCI training set
test_etpr = test_etpr.apply(add_etpr_question, axis=1)

In [40]:
print(test_etpr['ground_truth'].value_counts())
print(test_etpr['ground_truth_text'].value_counts())
print(test_etpr['question'].value_counts())
print(test_etpr['NACCUDSD'].value_counts())

ground_truth
G    762
E    760
F    700
D    694
H    690
B    687
A    669
C    664
Name: count, dtype: int64
ground_truth_text
Alzheimer's disease (AD)                                                                                                                                                                                             3219
Not applicable (no cognitive impairment)                                                                                                                                                                              951
Frontotemporal lobar degeneration and its variants, including primary progressive aphasia, corticobasal degeneration and progressive supranuclear palsy, and with or without amyotrophic lateral sclerosis (FTLD)     759
Lewy body disease (LBD)                                                                                                                                                                                               410

In [41]:
# Update set of used IDs to avoid reusing these patients elsewhere
etpr_ids = set(test_etpr['ID'])

In [42]:
# Select only the columns needed for modeling
test_etpr = test_etpr[columns]
test_etpr['Q_TYPE'] = 'Primary etiology'

In [45]:
test_etpr.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/with_summary/test_etpr.csv", index=False)

## NACCUDSD

In [46]:
# Filter and sample patients for etiology prediction task,
# excluding those already selected for NP, PET, DAT, or MCI tasks
test_cog = (
    test
    .sample(frac=1, random_state=42)  # Shuffle the sampled rows
    .reset_index(drop=True)
)

# Check sampled distribution
test_cog['NACCUDSD'].value_counts()

NACCUDSD
4    4369
1     951
3     492
Name: count, dtype: int64

In [47]:
import random, string

# Question variants for cognitive status identification
cog_question_variants = [
    "Using the information provided, determine the patient's cognitive status.",
    "Identify the patient's cognitive status based on the given information.",
    "Based on the provided clinical details, classify the patient's cognitive status.",
    "From the information available, determine the correct cognitive status for the patient.",
    "Analyze the patient's information and identify their cognitive status."
]

# Original answer texts
COG_OPTION_TEXTS = [
    "Normal Cognition (NC)",
    "Mild Cognitive Impairment (MCI)",
    "Dementia (DE)"
]

# Mapping from NACCUDSD to the correct label text
COG_ANSWER_MAP = {
    1: "Normal Cognition (NC)",
    3: "Mild Cognitive Impairment (MCI)",
    4: "Dementia (DE)",
}

def add_cog_question(row, *, seed=None):
    rng = random.Random(seed if seed is not None else row.name)

    # 1️⃣ Random question phrasing
    row['question'] = rng.choice(cog_question_variants)

    # 2️⃣ Shuffle the answer options
    shuffled = COG_OPTION_TEXTS.copy()
    rng.shuffle(shuffled)
    letters = list(string.ascii_uppercase[:len(shuffled)])
    row['options'] = "\n".join(f"{ltr}. {opt}" for ltr, opt in zip(letters, shuffled))

    # 3️⃣ Determine correct answer letter
    correct_text = COG_ANSWER_MAP.get(row['NACCUDSD'], None)
    if correct_text in shuffled:
        row['ground_truth'] = letters[shuffled.index(correct_text)]
    else:
        row['ground_truth'] = None  # fallback for unrecognized codes
    
    row['ground_truth_text'] = correct_text

    return row


In [48]:
# Apply the function to the MCI training set
test_cog = test_cog.apply(add_cog_question, axis=1)

In [49]:
print(test_cog['ground_truth'].value_counts())
print(test_cog['ground_truth_text'].value_counts())
print(test_cog['question'].value_counts())

ground_truth
B    2004
C    1918
A    1890
Name: count, dtype: int64
ground_truth_text
Dementia (DE)                      4369
Normal Cognition (NC)               951
Mild Cognitive Impairment (MCI)     492
Name: count, dtype: int64
question
Analyze the patient's information and identify their cognitive status.                     1191
Using the information provided, determine the patient's cognitive status.                  1188
From the information available, determine the correct cognitive status for the patient.    1177
Identify the patient's cognitive status based on the given information.                    1132
Based on the provided clinical details, classify the patient's cognitive status.           1124
Name: count, dtype: int64


In [50]:
# Update set of used IDs to avoid reusing these patients elsewhere
cog_ids = set(test_cog['ID'])

In [51]:
# Select only the columns needed for modeling
test_cog = test_cog[columns]
test_cog['Q_TYPE'] = 'Cognitive status'

In [52]:
x = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/with_summary/test_cog.csv")
test_cog.drop(["COHORT"], axis=1).equals(x)

True

In [53]:
test_cog.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/with_summary/test_cog.csv", index=False)

In [71]:
json.loads(test_cog.iloc[500]['patient_summary'])['Subject Medications']

{'Subject taking any medications': 'Yes',
 'Name of medications used within two weeks of UDS visit': 'CYANOCOBALAMIN, SIMVASTATIN, METFORMIN, QUETIAPINE, GALANTAMINE, MEMANTINE, DULOXETINE',
 'Reported current use of an antidepressant': 'Reported use at visit',
 "Reported current use of a FDA-approved medication for Alzheimer's disease symptoms": 'Reported use at visit',
 'Total number of medications reported at each visit (range: (0.0 - 40.0))': 7.0,
 'Reported current use of an antipsychotic agent': 'Reported use at visit',
 'Reported current use of a diabetes medication': 'Reported use at visit',
 'Reported current use of lipid lowering medication': 'Reported use at visit'}

In [64]:
c = [f'DRUG{i}' for i in range(1,41)]

In [72]:
test[test['ID'] == test_cog.iloc[500]['ID']][c].dropna(axis=1, how='any', inplace=False)

,DRUG1,DRUG2,DRUG3,DRUG4,DRUG5,DRUG6,DRUG7
4615,CYANOCOBALAMIN,SIMVASTATIN,METFORMIN,QUETIAPINE,GALANTAMINE,MEMANTINE,DULOXETINE
